# 第12章 深度学习与序列模型 —— 配套代码

对应正文 `book/part3/12-deep-learning.md`。

> **运行前准备**：请先在终端执行 `uv run python scripts/make_sample_data.py` 生成内置示例数据。

## 演示内容

1. 环境初始化与数据加载
2. 激活函数可视化（ReLU / Sigmoid / Tanh）
3. 金融序列特征工程
4. 滑动窗口构造序列样本（防前视）
5. 按时间切分训练/验证集 + 标准化（仅用训练集统计）
6. 定义 LSTM 模型（nn.Module）
7. 训练循环 + 打印 loss（含早停）
8. 训练/验证 loss 曲线
9. 验证集预测与评估
10. 与朴素基线对比（持久预测）
11. Dropout 效果对比（过拟合演示）
12. GRU vs LSTM 对比
13. 综合汇总
14. 习题参考解答


In [ ]:
# Cell 1：环境初始化与数据加载
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from fds import load_sample_prices, daily_returns, set_chinese_font

set_chinese_font()
torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device('cpu')  # 强制使用 CPU

prices = load_sample_prices()
rets   = daily_returns(prices)

print(f'价格数据维度：{prices.shape}')
print(f'收益率维度：{rets.shape}')
print(f'日期范围：{rets.index[0].date()} 至 {rets.index[-1].date()}')
print(f'股票列：{list(rets.columns)}')
print(f'PyTorch 版本：{torch.__version__}')


## 12.1 激活函数可视化

三种常用激活函数的形状与导数特性直接影响训练动态：

- **Sigmoid**：输出 $(0,1)$，两端梯度趋近 0（梯度消失）
- **Tanh**：输出 $(-1,1)$，零中心，梯度消失问题依然存在
- **ReLU**：正半轴梯度恒为 1，负半轴为 0（"死亡神经元"问题）


In [ ]:
# Cell 2：激活函数可视化
z = np.linspace(-4, 4, 300)
sigmoid_v = 1 / (1 + np.exp(-z))
tanh_v    = np.tanh(z)
relu_v    = np.maximum(0, z)

d_sigmoid = sigmoid_v * (1 - sigmoid_v)
d_tanh    = 1 - tanh_v ** 2
d_relu    = (z > 0).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(z, sigmoid_v, label='Sigmoid', color='#1f77b4', lw=2)
ax.plot(z, tanh_v,    label='Tanh',    color='#2ca02c', lw=2)
ax.plot(z, relu_v,    label='ReLU',    color='#d62728', lw=2)
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('z')
ax.set_ylabel('f(z)')
ax.set_title('激活函数')
ax.legend()
ax.set_ylim(-1.5, 1.5)

ax2 = axes[1]
ax2.plot(z, d_sigmoid, label="Sigmoid 导数", color='#1f77b4', lw=2)
ax2.plot(z, d_tanh,    label="Tanh 导数",    color='#2ca02c', lw=2)
ax2.plot(z, d_relu,    label="ReLU 导数",    color='#d62728', lw=2)
ax2.axhline(0, color='gray', lw=0.8, linestyle='--')
ax2.set_xlabel('z')
ax2.set_ylabel('梯度')
ax2.set_title('激活函数的导数（梯度）')
ax2.legend()
ax2.set_ylim(-0.1, 1.1)

plt.suptitle('常用激活函数：函数值 vs 梯度', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Sigmoid/Tanh 在 |z| > 2 时梯度趋近 0，深层网络训练困难（梯度消失）')
print('ReLU 正区间梯度恒为 1，是深层网络的默认选择')


## 12.2 金融序列特征工程

以 **TECH** 单股为实验对象，构造以下特征：

| 特征 | 含义 | 前视安全性 |
|------|------|----------|
| `ret_1` | 昨日收益率 | `.shift(1)` |
| `vol_5` | 5日滚动波动率 | `.shift(1)` |
| `mom_5` | 5日累计收益 | `.shift(1)` |

**预测目标**：未来5日实现波动率（回归任务），即 `rolling(5).std().shift(-5)`。
波动率的可预测性远高于收益率方向，是深度学习在金融中最适合的场景之一。


In [ ]:
# Cell 3：特征工程与目标变量构造
STOCK = 'TECH'
ret   = rets[STOCK]

feat = pd.DataFrame(index=ret.index)
# 所有特征均已 shift(1)，使用昨日数据预测今日目标 -> 防前视
feat['ret_1'] = ret.shift(1)
feat['vol_5'] = ret.shift(1).rolling(5).std()
feat['mom_5'] = ret.shift(1).rolling(5).sum()

# 目标：未来5日实现波动率
target = ret.rolling(5).std().shift(-5)

df = feat.copy()
df['target'] = target
df = df.dropna()

FEATURE_COLS = ['ret_1', 'vol_5', 'mom_5']
X_raw = df[FEATURE_COLS].values
y_raw = df['target'].values

print(f'可用样本数：{len(df)}')
print(f'特征维度：{X_raw.shape[1]}')
print(f'目标（波动率）均值：{y_raw.mean():.4f}，标准差：{y_raw.std():.4f}')
df[FEATURE_COLS + ['target']].describe().round(4)


## 12.3 滑动窗口构造序列样本（防前视）

LSTM 期望输入格式为 `(batch, seq_len, features)`，需要从一维时序构造三维序列样本。

滑动窗口大小 `LOOKBACK = 20`（约一个月的交易日），每次向前移动一步。

**关键**：样本 $i$ 使用第 $i-T$ 到 $i-1$ 步的特征，预测第 $i$ 步的目标，绝无前视。


In [ ]:
# Cell 4：滑动窗口构造序列样本
LOOKBACK = 20  # look-back 窗口大小（约一个月）

def make_sequences(X, y, T):
    """构造 (n_samples, T, n_features) 的序列样本。
    样本 i 使用 X[i-T:i] 预测 y[i]，严格无前视。
    """
    Xs, ys = [], []
    for i in range(T, len(X)):
        Xs.append(X[i - T:i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_seq, y_seq = make_sequences(X_raw, y_raw, LOOKBACK)

print(f'序列样本维度 X: {X_seq.shape}  (样本数, 窗口长度, 特征数)')
print(f'目标维度     y: {y_seq.shape}')

# 按时间切分（8:2），不能随机打乱！
split = int(len(X_seq) * 0.8)
X_train, X_val = X_seq[:split], X_seq[split:]
y_train, y_val = y_seq[:split], y_seq[split:]

print(f'\n训练集：{X_train.shape[0]} 条')
print(f'验证集：{X_val.shape[0]} 条')
print('（按时间前后切分，验证集完全在训练集之后）')


## 12.4 标准化（仅用训练集统计量）

> **警告**：必须先切分，再标准化。若在全数据集上 fit scaler，验证集的均值/方差信息会泄露到训练过程中，造成乐观的评估偏差。


In [ ]:
# Cell 5：标准化（仅用训练集统计量）
n_tr, T_lb, n_feat = X_train.shape
n_val = X_val.shape[0]

X_train_2d = X_train.reshape(-1, n_feat)
X_val_2d   = X_val.reshape(-1, n_feat)

scaler_X = StandardScaler()
scaler_X.fit(X_train_2d)  # 只在训练集上 fit！
X_train_sc = scaler_X.transform(X_train_2d).reshape(n_tr, T_lb, n_feat)
X_val_sc   = scaler_X.transform(X_val_2d).reshape(n_val, T_lb, n_feat)

scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_sc   = scaler_y.transform(y_val.reshape(-1, 1)).ravel()

X_tr_t = torch.tensor(X_train_sc, dtype=torch.float32)
X_vl_t = torch.tensor(X_val_sc,   dtype=torch.float32)
y_tr_t = torch.tensor(y_train_sc, dtype=torch.float32)
y_vl_t = torch.tensor(y_val_sc,   dtype=torch.float32)

print(f'训练张量：{X_tr_t.shape}')
print(f'验证张量：{X_vl_t.shape}')
print(f'目标均值（训练集标准化后）：{y_tr_t.mean():.4f}，标准差：{y_tr_t.std():.4f}')


## 12.5 定义 LSTM 模型

```
输入 (batch, 20, 3)
   -> LSTM(hidden=32, layers=1)
   -> 取最后时间步隐藏状态 (batch, 32)
   -> Dropout(0.2)
   -> Linear(32 -> 1)
输出 (batch,) 标准化后的波动率预测
```

参数量：$4 \times 32 \times (32 + 3 + 1) = 4608$，约 4.5K 参数——刻意保持小网络以控制过拟合。


In [ ]:
# Cell 6：定义 LSTM 模型
class VolLSTM(nn.Module):
    """单层 LSTM 波动率预测器。"""
    def __init__(self, input_dim, hidden_dim=32, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        last = self.dropout(last)
        return self.fc(last).squeeze(-1)


model = VolLSTM(input_dim=n_feat, hidden_dim=32, dropout=0.2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'模型结构：')
print(model)
print(f'\n总参数量：{total_params:,}')
print(f'理论参数量（单层 LSTM）：4 * 32 * (32 + {n_feat!r} + 1) = {4 * 32 * (32 + 3 + 1)}')


In [ ]:
# Cell 7：训练循环 + 早停
def train_model(model, X_tr, y_tr, X_vl, y_vl,
                lr=1e-3, weight_decay=1e-4,
                max_epochs=30, patience=8, batch_size=32,
                verbose=True):
    """带早停的 LSTM 训练函数，返回 (model, train_losses, val_losses)。"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    dataset   = TensorDataset(X_tr, y_tr)
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_weights  = copy.deepcopy(model.state_dict())
    wait = 0

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        train_loss = epoch_loss / len(X_tr)

        model.eval()
        with torch.no_grad():
            val_pred = model(X_vl.to(DEVICE))
            val_loss = criterion(val_pred, y_vl.to(DEVICE)).item()

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if verbose and (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1:3d}/{max_epochs}  '
                  f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f'早停于 epoch {epoch+1}（patience={patience}）')
                break

    model.load_state_dict(best_weights)
    return model, train_losses, val_losses


print('=== 训练 LSTM（Dropout=0.2，早停 patience=8）===')
model, train_losses, val_losses = train_model(
    model, X_tr_t, y_tr_t, X_vl_t, y_vl_t,
    max_epochs=30, patience=8
)


## 12.6 训练/验证 Loss 曲线

可视化训练过程：观察是否出现"剪刀差"（训练 loss 持续下降但验证 loss 上升），即过拟合的典型症状。


In [ ]:
# Cell 8：训练/验证 loss 曲线
epochs_list = range(1, len(train_losses) + 1)
best_epoch  = int(np.argmin(val_losses)) + 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs_list, train_losses, 'o-', color='steelblue', lw=2, ms=4, label='训练 Loss (MSE)')
ax.plot(epochs_list, val_losses,   's--', color='tomato',    lw=2, ms=4, label='验证 Loss (MSE)')
ax.axvline(best_epoch, color='navy', linestyle=':', lw=1.5,
           label=f'最优 epoch={best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss（标准化空间）')
ax.set_title('LSTM 训练过程：训练 vs 验证 Loss 曲线')
ax.legend()
plt.tight_layout()
plt.show()

print(f'最优 epoch：{best_epoch}')
print(f'最优验证 loss：{min(val_losses):.4f}')
print(f'最终训练 loss：{train_losses[-1]:.4f}')


In [ ]:
# Cell 9：验证集预测与评估
model.eval()
with torch.no_grad():
    y_pred_sc = model(X_vl_t).numpy()

y_pred = scaler_y.inverse_transform(y_pred_sc.reshape(-1, 1)).ravel()
y_true = y_val  # 原始空间真实值

rmse_lstm = np.sqrt(mean_squared_error(y_true, y_pred))
mae_lstm  = mean_absolute_error(y_true, y_pred)

# 朴素持久基线：用上一步的真实值预测当前步
y_naive    = np.concatenate([[y_train[-1]], y_true[:-1]])
rmse_naive = np.sqrt(mean_squared_error(y_true, y_naive))
mae_naive  = mean_absolute_error(y_true, y_naive)

print('=== 验证集评估结果 ===')
print(f'              RMSE           MAE')
print(f'LSTM      {rmse_lstm:.6f}   {mae_lstm:.6f}')
print(f'持久基线  {rmse_naive:.6f}   {mae_naive:.6f}')
print(f'\nLSTM vs 基线 RMSE 比：{rmse_lstm / rmse_naive:.3f}')
if rmse_lstm < rmse_naive:
    print('-> LSTM 优于持久基线')
else:
    print('-> LSTM 未显著优于持久基线（金融序列低信噪比的体现）')

# 可视化
val_start_idx = split + LOOKBACK
val_end_idx   = val_start_idx + len(y_true)
# 安全取日期
if val_end_idx <= len(df):
    val_dates = df.index[val_start_idx:val_end_idx]
else:
    val_dates = df.index[val_start_idx:]
n_plot = min(len(val_dates), len(y_true), len(y_pred))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(val_dates[:n_plot], y_true[:n_plot],  color='black',     lw=1.5, label='真实波动率', alpha=0.8)
ax.plot(val_dates[:n_plot], y_pred[:n_plot],  color='tomato',    lw=1.5, label='LSTM 预测', alpha=0.8)
ax.plot(val_dates[:n_plot], y_naive[:n_plot], color='steelblue', lw=1.2, label='持久基线', linestyle='--', alpha=0.7)
ax.set_xlabel('日期')
ax.set_ylabel('5日实现波动率')
ax.set_title(f'{STOCK} 未来5日波动率预测：LSTM vs 持久基线（验证集）')
ax.legend()
plt.tight_layout()
plt.show()


## 12.7 Dropout 效果对比：过拟合演示

训练一个**无 Dropout** 的版本，对比训练/验证 loss 曲线的分叉程度，直观展示 Dropout 对过拟合的抑制作用。

> 在低信噪比的金融数据上，即使是小网络也会出现过拟合：训练 loss 下降，验证 loss 停滞或上升。


In [ ]:
# Cell 10：Dropout 对比（有 vs 无）
print('=== 训练无 Dropout 的 LSTM（过拟合演示）===')
torch.manual_seed(0)
model_nodrop = VolLSTM(input_dim=n_feat, hidden_dim=32, dropout=0.0).to(DEVICE)
model_nodrop, tr_no, vl_no = train_model(
    model_nodrop, X_tr_t, y_tr_t, X_vl_t, y_vl_t,
    max_epochs=30, patience=30, verbose=True  # 关闭早停跑满
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, tr_l, vl_l, title in [
    (axes[0], train_losses, val_losses, 'Dropout=0.2（带早停）'),
    (axes[1], tr_no, vl_no, 'Dropout=0.0（无早停，满30轮）'),
]:
    ep = range(1, len(tr_l) + 1)
    ax.plot(ep, tr_l, 'o-', color='steelblue', lw=2, ms=3, label='训练 Loss')
    ax.plot(ep, vl_l, 's--', color='tomato',    lw=2, ms=3, label='验证 Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Dropout 效果对比：训练 vs 验证 Loss', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Dropout=0.2：最终训练 loss={train_losses[-1]:.4f}，最佳验证 loss={min(val_losses):.4f}')
print(f'Dropout=0.0：最终训练 loss={tr_no[-1]:.4f}，最终验证 loss={vl_no[-1]:.4f}')
print()
print('若无 Dropout，训练/验证 loss 出现明显分叉 -> 过拟合的典型症状')


## 12.8 GRU vs LSTM 对比

GRU 参数量约为 LSTM 的 75%，在小样本下往往表现相当甚至更好。

| 模型 | 门数 | 参数量（hidden=32, input=3） |
|------|------|-----------------------------|
| LSTM | 4（遗忘/输入/输出/候选）| $4 \times 32 \times (32+3+1) = 4608$ |
| GRU  | 3（更新/重置/候选）| $3 \times 32 \times (32+3+1) = 3456$ |


In [ ]:
# Cell 11：GRU vs LSTM 对比
class VolGRU(nn.Module):
    """单层 GRU 波动率预测器。"""
    def __init__(self, input_dim, hidden_dim=32, dropout=0.2):
        super().__init__()
        self.gru     = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        last   = out[:, -1, :]
        return self.fc(self.dropout(last)).squeeze(-1)


torch.manual_seed(0)
gru_model = VolGRU(input_dim=n_feat, hidden_dim=32, dropout=0.2).to(DEVICE)

print('=== 训练 GRU ===')
gru_model, tr_gru, vl_gru = train_model(
    gru_model, X_tr_t, y_tr_t, X_vl_t, y_vl_t,
    max_epochs=30, patience=8, verbose=True
)

gru_model.eval()
with torch.no_grad():
    y_pred_gru_sc = gru_model(X_vl_t).numpy()
y_pred_gru = scaler_y.inverse_transform(y_pred_gru_sc.reshape(-1, 1)).ravel()

rmse_gru = np.sqrt(mean_squared_error(y_true, y_pred_gru))
mae_gru  = mean_absolute_error(y_true, y_pred_gru)

print()
print('=== GRU vs LSTM vs 基线（验证集）===')
rows = [
    {'模型': '持久基线', 'RMSE': round(rmse_naive, 6), 'MAE': round(mae_naive, 6),
     '参数量': 0},
    {'模型': 'LSTM',    'RMSE': round(rmse_lstm, 6),  'MAE': round(mae_lstm, 6),
     '参数量': sum(p.numel() for p in model.parameters())},
    {'模型': 'GRU',     'RMSE': round(rmse_gru, 6),   'MAE': round(mae_gru, 6),
     '参数量': sum(p.numel() for p in gru_model.parameters())},
]
cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))


## 12.9 综合汇总

总结本章实验结果，并结合「深度学习在金融中的局限性」做定性分析。


In [ ]:
# Cell 12：综合汇总可视化
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左图：RMSE 对比
ax1 = axes[0]
models_r = cmp_df['模型'].tolist()
rmses    = cmp_df['RMSE'].tolist()
colors   = ['gray', '#1f77b4', '#ff7f0e']
bars = ax1.bar(models_r, rmses, color=colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, rmses):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
             f'{val:.5f}', ha='center', va='bottom', fontsize=9)
ax1.set_ylabel('RMSE（验证集，原始空间）')
ax1.set_title('波动率预测 RMSE 对比')
ax1.set_ylim(0, max(rmses) * 1.25)

# 右图：散点图（真实 vs 预测）
ax2 = axes[1]
n_sc = min(len(y_true), len(y_pred), len(y_pred_gru))
ax2.scatter(y_true[:n_sc], y_pred[:n_sc],     alpha=0.5, color='tomato',    s=20, label='LSTM')
ax2.scatter(y_true[:n_sc], y_pred_gru[:n_sc], alpha=0.5, color='steelblue', s=20, label='GRU', marker='^')
lims = [min(y_true[:n_sc].min(), y_pred[:n_sc].min(), y_pred_gru[:n_sc].min()),
        max(y_true[:n_sc].max(), y_pred[:n_sc].max(), y_pred_gru[:n_sc].max())]
ax2.plot(lims, lims, 'k--', lw=1, label='完美预测线')
ax2.set_xlabel('真实波动率')
ax2.set_ylabel('预测波动率')
ax2.set_title('真实 vs 预测波动率（验证集）')
ax2.legend(fontsize=9)

plt.suptitle('LSTM/GRU 波动率预测综合汇总', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('【综合结论】')
print('1. 金融波动率具有一定可预测性（波动率聚类效应），LSTM 可以捕捉部分信号')
print('2. 收益率方向预测（涨跌）因信噪比过低，深度模型通常无法稳健优于基线')
print('3. GRU 参数量更少，在 750 日的小样本上与 LSTM 表现相当')
print('4. 过拟合是核心风险：用 Dropout + 早停 + 权重衰减三重防线应对')
print('5. 严格的样本外测试是评估任何模型的唯一可信标准')


---

## 习题参考解答

以下各 cell 对应正文习题，可直接运行验证。


In [ ]:
# === 习题 12.1：LSTM 门控极端情形验证 ===
print('习题 12.1：LSTM 门控在极端情形下的行为')
print()

torch.manual_seed(0)
hidden_dim = 2
lstm_cell  = nn.LSTMCell(input_size=1, hidden_size=hidden_dim)

h = torch.zeros(1, hidden_dim)
c = torch.tensor([[0.5, -0.3]])
x = torch.tensor([[0.1]])

# f_t -> 1（全保留）：遗忘门偏置设为极大正数
with torch.no_grad():
    lstm_cell.bias_ih.data[hidden_dim:2*hidden_dim] = 10.0
    lstm_cell.bias_hh.data[hidden_dim:2*hidden_dim] = 10.0
_, c_keep = lstm_cell(x, (h, c))
print(f'初始记忆 c = {c[0].numpy()}')
print(f'遗忘门极大(f->1, 全保留) c_new = {c_keep[0].detach().numpy()}')
print('-> c_new 接近 c（历史几乎完全保留，类比趋势/动量市场）')
print()

# f_t -> 0（全遗忘）：遗忘门偏置设为极小负数
with torch.no_grad():
    lstm_cell.bias_ih.data[hidden_dim:2*hidden_dim] = -10.0
    lstm_cell.bias_hh.data[hidden_dim:2*hidden_dim] = -10.0
_, c_forget = lstm_cell(x, (h, c))
print(f'遗忘门极小(f->0, 全遗忘) c_new = {c_forget[0].detach().numpy()}')
print('-> c_new 仅由当前输入决定（历史清零，类比重大冲击后市场重置）')


In [ ]:
# === 习题 12.2：前视偏差检测 ===
print('习题 12.2：前视偏差检测与修正')
print()

stock_ret = rets['TECH']

feat_ok  = stock_ret.shift(1).rolling(5).std()   # 正确：只用前一日数据
feat_bad = stock_ret.rolling(5).std()              # 含当日数据（有争议）

corr_bad = feat_bad.corr(stock_ret)
corr_ok  = feat_ok.corr(stock_ret)

print(f'含当日数据的特征与当日收益率相关：{corr_bad:.4f}')
print(f'仅前一日数据的特征与当日收益率相关：{corr_ok:.4f}')
print()
print('【分析】')
print('feat_bad 含有当日收益率，若用于预测次日涨跌（shift(-1)）则无前视')
print('但若用于预测当日涨跌（不 shift 目标），则是严重的前视偏差')
print('最安全的做法：所有特征一律 shift(1)，目标 shift(-1) 或不 shift')


In [ ]:
# === 习题 12.3：GRU vs LSTM 参数量计算 ===
print('习题 12.3：GRU vs LSTM 参数量')
print()
print(f'{"input":>6} {"hidden":>6} {"LSTM":>8} {"GRU":>8} {"GRU/LSTM":>10}')
for d, h in [(3, 16), (3, 32), (5, 32), (10, 64)]:
    lstm_p = 4 * h * (h + d + 1)
    gru_p  = 3 * h * (h + d + 1)
    print(f'{d:>6} {h:>6} {lstm_p:>8} {gru_p:>8} {gru_p/lstm_p:>10.3f}')
print()
print('结论：GRU 参数量约为 LSTM 的 75%（= 3/4），小样本场景下过拟合风险更低')
print('A 股 3 年日频约 750 条，优先考虑 GRU；有高频分钟级数据时 LSTM 更有价值')


In [ ]:
# === 习题 12.4：标准化泄露演示 ===
print('习题 12.4："先标准化后切分" vs "先切分后标准化" 的差异')
print()

X2d = X_raw
sp  = int(len(X2d) * 0.8)

# 错误做法：先在全集 fit
sc_wrong = StandardScaler().fit(X2d)
# 正确做法：先切分，再在训练集 fit
sc_right = StandardScaler().fit(X2d[:sp])

mean_diff = sc_wrong.mean_ - sc_right.mean_
std_diff  = sc_wrong.scale_ - sc_right.scale_

print(f'均值差异（全集 - 训练集）：{mean_diff.round(6)}')
print(f'方差差异（全集 - 训练集）：{std_diff.round(6)}')
print()
print('即使差异看起来较小，在市场均值漂移时（如牛转熊）差异会显著放大')
print('"先标准化后切分" 将测试期统计信息写入 scaler，是方法论漏洞')


In [ ]:
# === 习题 12.5：简易 GARCH(1,1) vs LSTM ===
print('习题 12.5：GARCH(1,1) vs LSTM 波动率预测对比')
print()

def garch11(returns, omega=1e-6, alpha=0.05, beta=0.94):
    """GARCH(1,1) 递推条件标准差（固定参数演示）。"""
    n = len(returns)
    sigma2 = np.zeros(n)
    sigma2[0] = float(np.var(returns))
    for t in range(1, n):
        sigma2[t] = omega + alpha * returns[t-1]**2 + beta * sigma2[t-1]
    return np.sqrt(sigma2)

stock_r = rets['TECH'].values
tr_start = LOOKBACK
tr_end   = tr_start + split
vl_end   = tr_end + len(y_true)
vl_end   = min(vl_end, len(stock_r))

garch_sigma = garch11(stock_r[:vl_end])
garch_val   = garch_sigma[tr_end:vl_end]

n_m = min(len(garch_val), len(y_true))
rmse_garch = np.sqrt(mean_squared_error(y_true[:n_m], garch_val[:n_m]))
mae_garch  = mean_absolute_error(y_true[:n_m], garch_val[:n_m])

print(f'              RMSE           MAE')
print(f'GARCH(1,1) {rmse_garch:.6f}   {mae_garch:.6f}')
print(f'LSTM       {rmse_lstm:.6f}   {mae_lstm:.6f}')
print(f'持久基线   {rmse_naive:.6f}   {mae_naive:.6f}')
print()

fig, ax = plt.subplots(figsize=(11, 4))
xax = range(n_m)
ax.plot(xax, y_true[:n_m],       color='black',   lw=1.5, label='真实波动率')
ax.plot(xax, garch_val[:n_m],    color='purple',  lw=1.5, linestyle='-.', label='GARCH(1,1)')
ax.plot(xax, y_pred[:n_m],       color='tomato',  lw=1.5, linestyle='--', label='LSTM')
ax.set_xlabel('验证集时间步')
ax.set_ylabel('5日实现波动率')
ax.set_title('波动率预测：GARCH(1,1) vs LSTM vs 真实值')
ax.legend()
plt.tight_layout()
plt.show()

print('结论：')
print('GARCH(1,1) 是强基线，参数仅 3 个，有完整的统计推断框架')
print('LSTM 在小样本下不一定优于 GARCH，需要更多数据和精心调参')
print('建议：将 GARCH 条件方差作为 LSTM 的输入特征之一（混合模型）')
